In [ ]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()

# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill

In [ ]:
llm= EasyLLM(provider="openai_responses")
agent=BasicAgent(name="test_skill", llm=llm,verbose_thinking=True)
agent.with_skill(CalculatorSkill())

In [ ]:
#自定义skill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""
agent.with_skill(TranslateSkill())


In [ ]:
from openai.types.shared import reasoning


agent.clear_history()
print(agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里"))

In [ ]:
manager=agent.skill_manager
prompt=manager.build_skills_prompt()
print(prompt)

In [ ]:
from skill.registry import SkillRegistry
skill_manage=SkillRegistry()
registered_names = skill_manage.discover_from_directory("./test_skills/")


In [ ]:
print(skill_manage.list_available())

In [ ]:
from skill.folder_loader import FolderSkillLoader
c_skill=FolderSkillLoader.load("./real_skills/crypto_skill/")

In [ ]:
print(c_skill.get_prompt())

In [ ]:
skill_manage.discover_from_directory("./real_skills/")

In [ ]:
print(skill_manage.list_available())


In [ ]:
crypto_skill=skill_manage.create('crypto_skill')
agent.with_skill(crypto_skill)
print(agent.get_enhanced_prompt())

In [ ]:
agent.invoke("i am a boy from china的 SHA-256 哈希值是什么")

In [ ]:
from memory.V2.WorkingMemory import WorkingMemory
from memory import MemoryConfig,MemoryManage
from memory.V2.Embedding.HuggingfaceEmbeddingModel import HuggingfaceEmbeddingModel
config = MemoryConfig(max_capacity=20)
working_memory = WorkingMemory(config)
mm = MemoryManage(
            config=config,
            user_id="test_integration_user",
            enable_working=True,
            working_memory=working_memory,
            enable_episodic=False,
            enable_semantic=False,
            enable_perceptual=False,
        ) 

In [ ]:
agent.with_memory(mm)
print(agent.get_enhanced_prompt())

In [5]:
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill

# 1. 把所有 Skill 注册到全局 Registry（启动时一次性完成）
registry = SkillRegistry.instance()
registry.register_class(CalculatorSkill)
# 为搜索提供元信息
registry.update_metadata("calculator", description="数学计算工具", tags=["math", "compute"])
registry.discover_from_directory("./real_skills/")
# 也可以从目录批量发现
# registry.discover_from_directory("./skills/")

# 2. 创建 Agent（不预加载任何 Skill）
agent1 = BasicAgent(name="assistant", llm=llm, verbose_thinking=True)
agent1.with_skill(MetaSkill(registry,manager=agent1.skill_manager))
print(agent1.get_enhanced_prompt())

INFO:skill.registry:从目录 './real_skills/' 发现并注册 1 个 Skill: ['crypto_skill']
INFO:agent.BasicAgent:BasicAgent 'assistant' 初始化完成，工具调用: 禁用，异步执行: 禁用，provider: openai_responses
INFO:skill.manager:📦 注册 Skill 'meta_skill' (v1.0.0)
INFO:skill.manager:✅ 激活 Skill 'meta_skill' (工具: ['skill_discovery_tool', 'load_skill_tool', 'unload_skill_tool'])


你是一个智能助手，具备使用工具解决问题的能力。

            ## 核心原则
            1. **先思考，再行动**：在调用工具前，先分析用户需求，确定是否需要使用工具
            2. **选择合适的工具**：根据任务需求选择最适合的工具
            3. **正确传递参数**：确保传递给工具的参数格式正确、内容准确
            4. **处理工具结果**：根据工具返回的结果并分析，继续推理或给出最终答案
            5. 在申请工具调用或者回复的同时，需要给出思考过程
            ## 工具使用指南
            - 当用户问题可以直接回答时，不必使用工具
            - 当需要获取实时信息、执行计算或操作外部系统时，使用工具
            - 可以连续调用多个工具来完成复杂任务
            - 如果工具调用失败，分析原因并尝试其他方案
            - 当收集到足够的信息后回答用户问题

            ## 可用工具
            [{'type': 'tool', 'name': 'skill_discovery_tool', 'description': '获取所有可用的额外技能包(Skill)列表及描述。当你发现当前工具箱中没有合适的工具时，调用此工具获取可加载的技能列表。', 'parameters': {'properties': {}, 'title': 'SkillDiscoveryParams', 'type': 'object'}}, {'type': 'tool', 'name': 'load_skill_tool', 'description': '加载一个技能包(Skill)到当前工具箱。请先通过 skill_discovery_tool 获取可用 Skill，然后使用返回的 name 调用此工具加载。', 'parameters': {'properties': {'skill_name': {'description': '要加载的 Skill 注册名称（从 skill_discovery_tool 的返回结果中获取）', 'title': 'Skill Name', 'ty

In [ ]:
agent1.invoke("i am a boy from china的 SHA-256 哈希值是什么")

{'calculator': '数学计算工具', 'crypto_skill': '提供密码学和哈希计算能力'}
